# 🛡️ CkCk Hoax Detection AI — Inference Pipeline

**Track B: The Privacy Brain (NLP / Generative AI)**

Clean inference script dengan PII filtering terintegrasi. Berjalan **100% offline**, CPU-only.

---

**Pipeline:**
```
Input (caption / frame / video)
        ↓
  prepare_input()      ← caption: langsung | frame/video: EasyOCR → fallback <15 char
        ↓
  PIIFilter.filter()   ← sensor NIK, telepon, email, rekening (Constraint B-3, B-4)
        ↓
  TextPreprocessor     ← normalisasi teks Indonesia
        ↓
  run_classifier()     ← ONNX inference, input numpy (Constraint B-2)
        ↓
  compute_confidence() ← softmax numpy → skor probabilitas
        ↓
  compute_support_score() ← rule-based pola manipulatif (penjelas hasil)
```

## 1. Setup & Imports

In [ ]:
import os
import sys
import time
import pandas as pd

sys.path.insert(0, os.path.abspath('.'))

from src.utils import load_config
from src.pii_filter import PIIFilter
from src.preprocessing import TextPreprocessor
from src.manipulative_detector import compute_support_score

# ── Pilih mode: ONNX (produksi) atau Mock (testing paralel) ──────────────────
# Ganti USE_MOCK = False setelah model ONNX selesai dikerjakan AI engineer.
USE_MOCK = not os.path.exists('models/indobert_classifier.onnx')

if USE_MOCK:
    print('⚠️  File ONNX belum ada — menggunakan MOCK inferencer.')
    print('   Jalankan training.ipynb → cell Export ke ONNX terlebih dahulu.')
    from src.mock_inferencer import load_models, prepare_input, run_classifier, compute_confidence, run_ckck_inference
else:
    print('✅ File ONNX ditemukan — menggunakan inferencer nyata.')
    from src.inferencer import load_models, prepare_input, run_classifier, compute_confidence, run_ckck_inference

config = load_config('config.yaml')
print(f'✅ Modules loaded (offline, no API calls)')
print(f'   Mode: {"MOCK" if USE_MOCK else "ONNX"}  |  Config: config.yaml')

## 2. Load Model & Inisialisasi Komponen

In [ ]:
# Load ONNX model + tokenizer dari lokal (Constraint B-2)
models = load_models(config)

# PII Filter — wajib di dalam pipeline (Constraint B-3, B-4)
pii_filter = PIIFilter(
    mask_char     = config['pii_filter']['mask_char'],
    enabled_types = config['pii_filter']['types'],
)

# Text preprocessor
preprocessor = TextPreprocessor(use_stemmer=False)

# OCR config dari config.yaml
ocr_cfg       = config.get('ocr', {})
OCR_MIN_CHARS = ocr_cfg.get('min_chars', 15)
OCR_LANGUAGES = ocr_cfg.get('languages', ['id', 'en'])

print(f'✅ Model loaded: {models["model_path"]}')
print(f'✅ Tokenizer  : {models["tokenizer_path"]}')
print(f'✅ PII Filter : {config["pii_filter"]["types"]}')
print(f'✅ OCR min    : {OCR_MIN_CHARS} karakter | languages: {OCR_LANGUAGES}')

## 3. Pipeline Function

In [ ]:
def run_inference(raw_input, input_type='caption', caption_fallback=None, verbose=True):
    """
    Wrapper tipis di atas run_ckck_inference untuk notebook ini.
    Menggunakan komponen yang sudah diinisialisasi di cell 2.
    """
    return run_ckck_inference(
        raw_input        = raw_input,
        input_type       = input_type,
        models           = models,
        pii_filter       = pii_filter,
        preprocessor     = preprocessor,
        support_scorer   = compute_support_score,
        caption_fallback = caption_fallback,
        ocr_min_chars    = OCR_MIN_CHARS,
        verbose          = verbose,
    )

print('✅ run_inference() siap digunakan.')

## 4. Demo — Caption (Teks Langsung)

In [ ]:
caption_samples = [
    # Valid
    'Pemerintah Indonesia mengumumkan kebijakan ekonomi baru untuk mendorong pertumbuhan investasi.',

    # Hoaks klasik
    'BREAKING!! Vaksin COVID-19 terbukti mengandung microchip 5G!! Bagikan sebelum dihapus!!',

    # Teks dengan PII — wajib disensor sebelum masuk model
    'Korban bernama Budi (NIK 3201234506780001, email budi@gmail.com) melaporkan penipuan.',

    # Pesan berantai
    'AWAS!! Modus penipuan baru!! Transfer ke rekening BCA 1234567890123456!! SEBARKAN ke semua teman!!',
]

print('🛡️ CkCk Hoax Detection — Demo Caption\n')
results = []
for text in caption_samples:
    r = run_inference(text, input_type='caption', verbose=True)
    results.append(r)

## 5. Demo — Frame (Gambar / OCR)

In [ ]:
# Ganti frame_path dengan path gambar nyata untuk demo
# Format yang didukung: JPG, PNG, WEBP, BMP

frame_path = 'test_data/sample_frame.jpg'   # ← ubah ke path gambar kamu

if os.path.exists(frame_path):
    print(f'📸 Menjalankan OCR pada: {frame_path}')
    r_frame = run_inference(
        raw_input        = frame_path,
        input_type       = 'frame',
        caption_fallback = 'Teks caption dari postingan media sosial (fallback)',
        verbose          = True,
    )
    print(f'OCR result    : {r_frame["ocr_result"]}')
    print(f'Fallback used : {r_frame["fallback_used"]}')
else:
    print(f'⚠️  File tidak ditemukan: {frame_path}')
    print('   Untuk demo OCR, letakkan file gambar di test_data/sample_frame.jpg')
    print('   atau ubah frame_path di atas.')

## 6. Demo — Video (Ekstraksi Frame + OCR)

In [ ]:
# Ganti video_path dengan path video nyata untuk demo
# Format yang didukung: MP4, AVI, MKV, MOV

video_path = 'test_data/sample_video.mp4'   # ← ubah ke path video kamu

if os.path.exists(video_path):
    print(f'🎬 Menjalankan OCR pada video: {video_path}')
    r_video = run_inference(
        raw_input        = video_path,
        input_type       = 'video',
        caption_fallback = 'Caption video dari TikTok/YouTube (fallback)',
        verbose          = True,
    )
    print(f'OCR result    : {r_video["ocr_result"]}')
    print(f'Fallback used : {r_video["fallback_used"]}')
else:
    print(f'⚠️  File tidak ditemukan: {video_path}')
    print('   Untuk demo OCR video, letakkan file video di test_data/sample_video.mp4')

## 7. Batch Inference dari Test Set

In [ ]:
test_path = config['data']['test_path']

if os.path.exists(test_path):
    test_df = pd.read_csv(test_path)
    print(f'Running batch inference on {len(test_df)} test samples...\n')

    batch_results = []
    for _, row in test_df.iterrows():
        r = run_inference(str(row['text']), input_type='caption', verbose=False)
        r['true_label'] = int(row['label'])  # untuk evaluasi
        batch_results.append(r)

    # Ringkasan
    total_time = sum(r['inference_time_ms'] for r in batch_results)
    avg_time   = total_time / len(batch_results)
    hoax_count = sum(1 for r in batch_results if r['prediction'] == 'HOAX')
    pii_total  = sum(r['pii_detected'] for r in batch_results)

    print(f'📊 Batch Results:')
    print(f'   Total samples     : {len(batch_results)}')
    print(f'   Prediksi HOAX     : {hoax_count}')
    print(f'   Prediksi VALID    : {len(batch_results) - hoax_count}')
    print(f'   Total PII ditemukan: {pii_total}')
    print(f'   Avg inference time: {avg_time:.1f}ms per sample')
    print(f'   Total time        : {total_time:.1f}ms')
else:
    print(f'⚠️  Test data tidak ditemukan di {test_path}')

## 8. Constraint Compliance Verification

In [ ]:
print('=' * 55)
print('CONSTRAINT COMPLIANCE — Track B: The Privacy Brain')
print('=' * 55)

# B-1: Model size
# IndoBERT-base-p2 ≈ 124M parameter — jauh di bawah batas 4B
# Diverifikasi saat training.ipynb (cell Model Setup)
print('\n[B-1] Model ≤ 4B parameter')
print('      IndoBERT-base-p2 ≈ 124M parameter ✅')

# B-2: Offline Total
print('\n[B-2] Offline Total')
print(f'      ONNX Provider : CPUExecutionProvider ✅')
print(f'      Tokenizer     : local_files_only=True ✅')
print(f'      OCR           : EasyOCR GPU=False (CPU-only) ✅')
print(f'      Zero network  : Tidak ada API call dalam pipeline ✅')

# B-3: PII Filter dalam pipeline
print('\n[B-3] PII Filter terintegrasi dalam pipeline')
pii_test = pii_filter.filter('NIK 3201234506780001 tel 081234567890')
print(f'      Test: {pii_test["pii_count"]} PII ditemukan')
print(f'      Filter berjalan SEBELUM classifier ✅')

# B-4: Cakupan PII
print(f'\n[B-4] PII Coverage: {len(config["pii_filter"]["types"])} types')
required = {'nik', 'phone', 'email', 'bank_account'}
enabled  = set(config['pii_filter']['types'])
for t in config['pii_filter']['types']:
    tag = '(wajib)' if t in required else '(bonus)'
    print(f'      → {t:15s} {tag} ✅')

# B-5: Fine-tuning
model_dir = os.path.join(config['paths']['model_dir'], 'best_model')
onnx_path = config['inference']['model_path']
finetuned = os.path.exists(model_dir) or os.path.exists(onnx_path)
print(f'\n[B-5] Fine-tuning Lokal: {"✅ PASS" if finetuned else "⚠️  Belum (jalankan training.ipynb)"}')

print('\n' + '=' * 55)
print('Mode saat ini:', '⚠️  MOCK (model ONNX belum ada)' if USE_MOCK else '✅ ONNX NYATA')
print('=' * 55)

---

**✅ Inference pipeline selesai.**

Untuk menggunakan dengan model ONNX nyata:
1. Jalankan `training.ipynb` hingga selesai
2. Jalankan cell **Export ke ONNX** di `training.ipynb`
3. Restart kernel notebook ini — `USE_MOCK` akan otomatis `False`